# 07 — Run the federation and collect the results

This notebook drives the twelve federated experiments and collects all thirteen
results into the table the thesis reports.

## Why this one notebook does not hold all of its own logic

Every other notebook in this project inlines what it does. This one cannot, and the
reason is worth being precise about rather than apologising for.

NVFLARE runs here in **ProdEnv**, which is a real deployment and not a simulator.
The server and each hospital are separate operating-system processes, each started
with its own X.509 identity, talking over mutual TLS. The training code is shipped
to each site and executed there by the FLARE runtime, in a process this notebook's
kernel does not own and cannot reach.

So the client training script has to be an importable module on disk. A function
defined in a notebook cell exists only in this kernel, and the hospital process
cannot import it. That is a property of the deployment, and it is the deployment
that makes the thesis's claim about a working system meaningful in the first place.

**Where the code actually lives, and what each file does**

| file | what it does |
|---|---|
| `src/federated/federation/client.py` | the per-hospital trainer. Receives global weights, trains one local epoch, evaluates on its own validation split, sends the update back. Also where the FedProx proximal term is applied. |
| `src/federated/federation/recipes.py` | turns one experiment row into an NVFLARE recipe. Passes the model as a built `nn.Module`, not a dotted path. |
| `src/federated/config/experiments.py` | the single declarative table of all thirteen experiments, the shared hyperparameters, and the six partitions. |
| `src/federated/common/` | model, data, training and evaluation, shared verbatim with the centralised baseline. |
| `src/scripts/start_federation.sh` | provisions the PKI and starts the server and hospital processes. |

The cells below read those files and print the parts that matter, so you can see
the configuration and the client loop without leaving the notebook.

**The one rule that shapes all of it.** The model is passed to the recipe as a
built object rather than as a dotted path plus keyword arguments. This project once
shipped a server building a ResNet-18 while every client built a ResNet-50. The path
resolved, the defaults differed, the run completed, and the numbers were
meaningless.

## Configuration

Paths, the protocol, and which experiments to run.

The training hyperparameters are not repeated here. They live in
`src/federated/config/experiments.py` and are the same object the centralised
baseline uses, which is what makes the comparison a comparison of federation. The
cell after this one prints them so you can check that without opening the file.

In [ ]:
from pathlib import Path

REPO_ROOT = Path.cwd().parent
SRC = REPO_ROOT / "src"

# INPUT, must exist. Built by notebook 06.
DATA_DIR = REPO_ROOT / "deployment" / "data"
GLOBAL_DIR = DATA_DIR / "global"              # the held-out test set, 268 patients
PARTITIONS_DIR = DATA_DIR / "partitions"      # one folder per scenario

# The deployment. `workspace/` holds the provisioned PKI, one folder per identity.
DEPLOYMENT = REPO_ROOT / "deployment"
WORKSPACE_DIR = DEPLOYMENT / "workspace"      # nvflare provision -w workspace
PROJECT_YML = DEPLOYMENT / "project.yml"      # the provisioning file
LOGS_DIR = DEPLOYMENT / "logs"                # one folder per test, one file per site

# OUTPUT. One folder per experiment, plus the summary CSV the thesis tables use.
RESULTS_DIR = REPO_ROOT / "results" / "federated"
SUMMARY_CSV = RESULTS_DIR / "all_experiments.csv"

# The scripts that drive the deployment.
START_FEDERATION = SRC / "scripts" / "start_federation.sh"
STOP_FEDERATION = SRC / "scripts" / "stop_federation.sh"
RUN_EXPERIMENT = SRC / "scripts" / "run_experiment.py"
RUN_CENTRALIZED = SRC / "scripts" / "run_centralized.py"
COLLECT_RESULTS = SRC / "scripts" / "collect_results.py"

# --------------------------------------------------------------------------- #
# THE PROTOCOL
# --------------------------------------------------------------------------- #

NUM_ROUNDS = 30           # 30 rounds x 1 local epoch = the same 30 epochs of data
LOCAL_EPOCHS = 1          # the centralised baseline sees. Matching the budget is
                          # what makes the gap federation rather than budget.
CENTRALIZED_EPOCHS = 30

FEDPROX_MU = 0.01         # the proximal coefficient. 0.01 is the value used
                          # throughout the cross-silo medical imaging literature
                          # and what NVFLARE defaults to. It was never tuned here,
                          # and the thesis says so.

KEY_METRIC = "val_balanced_accuracy"   # what the SERVER selects the global model on.
                                       # It must come from held-out client data. An
                                       # earlier iteration selected on training
                                       # accuracy and therefore picked whichever
                                       # model let clients memorise their shard best.

# Which experiments to run. test01 is the centralised baseline and is not an
# NVFLARE job; it goes through run_centralized.py instead.
TO_RUN = ["test02", "test03", "test04", "test05", "test06", "test07",
          "test08", "test09", "test10", "test11", "test12", "test13"]
# TO_RUN = ["test10", "test11"]       # just the RQ2 cohort pair
# TO_RUN = []                          # collect only, run nothing

DRY_RUN = True            # True prints what would be submitted and submits nothing.
                          # Leave it True until you have read the plan.

print(f"data        {DATA_DIR}     exists: {DATA_DIR.is_dir()}")
print(f"workspace   {WORKSPACE_DIR}  exists: {WORKSPACE_DIR.is_dir()}")
print(f"results     {RESULTS_DIR}")
print(f"to run      {len(TO_RUN)} experiments, dry run = {DRY_RUN}")

## Imports, and the shared configuration

`src/federated/` is put on the path so the declarative experiment table can be read.
That table is the single source of truth for all thirteen rows, and reading it here
rather than restating it is deliberate: a number copied into a notebook is a number
that will drift.

In [ ]:
import json
import subprocess
import sys
from dataclasses import asdict

import numpy as np
import pandas as pd

sys.path.insert(0, str(SRC / "federated"))
from config import experiments as EX     # noqa: E402

print("TRAINING — identical for the centralised baseline and every hospital")
print("-" * 70)
for k, v in asdict(EX.TRAINING).items():
    print(f"  {k:<36} {v}")
print()
print("FEDERATION — the protocol")
print("-" * 70)
for k, v in asdict(EX.FEDERATION).items():
    print(f"  {k:<36} {v}")

## The thirteen experiments

One centralised baseline against twelve federated runs. The federated ones vary the
number of hospitals and how the data is divided, and each configuration runs once
with FedAvg and once with FedProx with everything else held fixed.

Nine experiments differing along three axes is exactly the shape of problem where
hand-written per-experiment configs drift apart. In the previous iteration of this
project three separate bugs came from that drift, and all three were invisible until
the results looked wrong. Here every experiment is a row.

The research question mapping:

| | |
|---|---|
| RQ1, can federated match centralised? | test01 against tests 02 to 13 |
| RQ2, what does heterogeneity cost? | test10 against test12, test11 against test13 |
| RQ3, FedAvg against FedProx | every even and odd pair. The one that matters is 10 against 11, where FedProx has real heterogeneity to correct |
| RQ4, what mitigates the limitations? | test09 and test11, plus the security measures |

In [ ]:
table = pd.DataFrame([{
    "id": e.id, "name": e.name, "kind": e.kind,
    "algorithm": e.algorithm or "-", "partition": e.partition or "-",
    "hospitals": e.n_clients, "research_question": e.research_question,
} for e in EX.EXPERIMENTS])
print(table.to_string(index=False))

## The client, read from disk

The per-hospital trainer. This is the code the FLARE runtime executes inside each
hospital process, and the cell below prints its documentation rather than restating
it, so what you read is what runs.

The one part worth understanding without opening the file is the FedProx term.
FedProx does not change how updates are combined on the server, which is why it
shares a recipe with FedAvg. It changes what each client optimises, by adding
`mu/2 * ||w_local - w_global||^2` to the local loss so a site cannot drift too far
from the model it was given.

That term lives entirely on the client, which creates a specific failure mode: a
client that receives the coefficient and ignores it is running FedAvg while the
results table says FedProx, and nothing would warn you. Two guards exist for that.
The recipe refuses to build a FedProx experiment with `mu <= 0`, and the client
refuses to apply a proximal term without the global weights to anchor to.

In [ ]:
CLIENT_PY = SRC / "federated" / "federation" / "client.py"
RECIPES_PY = SRC / "federated" / "federation" / "recipes.py"

for path in (CLIENT_PY, RECIPES_PY):
    text = path.read_text()
    doc = text.split('"""')[1] if '"""' in text else "(no module docstring)"
    print("=" * 78)
    print(f"{path.relative_to(REPO_ROOT)}   ({len(text.splitlines())} lines)")
    print("=" * 78)
    print(doc.strip())
    print()

## Preflight

What must be true before a single job is submitted. Every one of these has a
failure mode that produces a completed run with a wrong number rather than an
error, which is why they are checked rather than assumed.

In [ ]:
def preflight(exp_ids):
    """Everything that must exist before submitting. Returns a list of problems."""
    problems = []

    if not (GLOBAL_DIR / "test.csv").is_file():
        problems.append(f"no global test set at {GLOBAL_DIR} — run notebook 06")

    for eid in exp_ids:
        exp = next((e for e in EX.EXPERIMENTS if e.id == eid), None)
        if exp is None:
            problems.append(f"{eid}: not in the experiment table")
            continue
        if exp.kind != "federated":
            continue
        part_dir = PARTITIONS_DIR / exp.partition
        if not (part_dir / "partition.json").is_file():
            problems.append(f"{eid}: partition {exp.partition} not built — notebook 06")
            continue
        meta = json.loads((part_dir / "partition.json").read_text())
        if meta["n_clients"] != exp.n_clients:
            problems.append(f"{eid}: partition has {meta['n_clients']} sites, "
                            f"experiment wants {exp.n_clients}")
        # A FedProx row with mu = 0 is FedAvg wearing the wrong label.
        if exp.algorithm == "fedprox" and EX.FEDERATION.fedprox_mu <= 0:
            problems.append(f"{eid}: declared FedProx but fedprox_mu is "
                            f"{EX.FEDERATION.fedprox_mu}")

    if not WORKSPACE_DIR.is_dir():
        problems.append(f"no provisioned workspace at {WORKSPACE_DIR}. "
                        f"Run: nvflare provision -p {PROJECT_YML}")
    return problems


issues = preflight(TO_RUN)
if issues:
    print("PREFLIGHT FAILED")
    for p in issues:
        print(f"  - {p}")
else:
    print("preflight passed — every partition, the global test set and the PKI are in place")

## Start the federation

This provisions the PKI if it does not exist and starts the server and one process
per hospital, each with its own certificate. It is a shell script rather than a
notebook cell because the processes have to outlive this kernel: a hospital started
from a cell would die when the kernel restarts, halfway through a 30-round run.

Each site writes its own log under `deployment/logs/`, which together with the admin
log is the record a run is reconstructed from months later.

Run it in a terminal, not here. The cell prints the command.

In [ ]:
print("Start the federation in a terminal, then come back:\n")
print(f"    bash {START_FEDERATION.relative_to(REPO_ROOT)}\n")
print("It provisions the PKI on first run, then starts one process per identity:")
print("  server, hospital_1 .. hospital_N, and the admin.")
print("Each writes its own log under deployment/logs/.\n")
print(f"To stop it:  bash {STOP_FEDERATION.relative_to(REPO_ROOT)}\n")

# Is anything listening? A crude but effective check that the server is up.
try:
    out = subprocess.run(["pgrep", "-fl", "nvflare"], capture_output=True, text=True,
                         timeout=10).stdout.strip()
    print("running NVFLARE processes:" if out else "no NVFLARE process found")
    print(out if out else "  start the federation before submitting anything")
except Exception as exc:
    print(f"could not check for running processes: {exc}")

## The centralised baseline

test01 is not an NVFLARE job. It is one machine holding all the data, and it goes
through `run_centralized.py`, which uses the same `TrainingConfig` object the
hospitals use.

Notebook 03 runs the same logic interactively. The difference is only where the
output lands and that this path is the one the reported number came from.

In [ ]:
cmd = [sys.executable, str(RUN_CENTRALIZED), "--seed", str(EX.TRAINING.seed),
       "--epochs", str(EX.FEDERATION.centralized_epochs)]

print("centralised baseline:\n")
print("    " + " ".join(str(c) for c in cmd) + "\n")

if DRY_RUN:
    print("DRY_RUN is True — not launched.")
else:
    result = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"exit code {result.returncode}")

## Submit the federated experiments

Each one is submitted through the admin API, using the admin identity's own
certificate, exactly as a coordinating centre would. Nothing writes a job config by
hand: the recipe builds it, because editing generated JSON is how this project once
ended up with a server building a different architecture from its clients.

A 30-round run at four hospitals takes roughly 15 minutes on a rented GPU. Twelve of
them plus the reruns is where the 102 GB of communication in the thesis comes from.

Leave `DRY_RUN` on for the first pass and read the plan.

In [ ]:
def submit(experiment_id, dry_run=True):
    """Submit one experiment and wait for it."""
    cmd = [sys.executable, str(RUN_EXPERIMENT), experiment_id]
    if dry_run:
        cmd.append("--dry-run")
    print("=" * 78)
    print(f"{experiment_id}   " + " ".join(str(c) for c in cmd[1:]))
    print("=" * 78)
    result = subprocess.run(cmd, cwd=REPO_ROOT)
    return result.returncode


if issues:
    print("preflight did not pass — nothing submitted")
elif not TO_RUN:
    print("TO_RUN is empty — nothing to submit")
else:
    for eid in TO_RUN:
        rc = submit(eid, dry_run=DRY_RUN)
        if rc != 0 and not DRY_RUN:
            print(f"{eid} exited {rc} — stopping so the failure is not buried")
            break

## Collect the results

Every finished experiment is re-scored on the same global test set, through one
code path. That last part matters: the centralised run already evaluated itself, and
it is re-scored here anyway so that every row in the table is produced by the same
function rather than by two functions that agree today.

For a federated experiment this loads the aggregated global model the server
selected. For the centralised one it loads the checkpoint chosen on validation AUC.

In [ ]:
cmd = [sys.executable, str(COLLECT_RESULTS), "--out", str(SUMMARY_CSV)]
print("    " + " ".join(str(c) for c in cmd) + "\n")

if DRY_RUN:
    print("DRY_RUN is True — not launched. The existing table is read below.")
else:
    result = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f"exit code {result.returncode}")

## The result

One centralised run against twelve federated ones, all scored on the same 268
patients.

Read it against the noise floor of 0.067 macro AUC, which is how far apart two runs
of a byte-identical configuration were measured. The question RQ1 asks is not
whether the federated numbers are higher, it is whether they land in the same place,
and that is what the count at the bottom answers.

In [ ]:
NOISE_FLOOR = 0.067

if not SUMMARY_CSV.is_file():
    print(f"{SUMMARY_CSV} not found — run the collection step above")
else:
    fed = pd.read_csv(SUMMARY_CSV)
    cols = [c for c in ["experiment", "kind", "algorithm", "n_clients", "partition",
                        "test_auc", "test_acc", "test_bal", "test_macro_f1"]
            if c in fed]
    print(fed[cols].to_string(index=False))

    central = fed[fed.kind == "centralized"].test_auc.iloc[0]
    federated = fed[fed.kind == "federated"].test_auc

    print(f"\ncentralised              {central:.4f}")
    print(f"federated mean of {len(federated):<2}     {federated.mean():.4f}")
    print(f"gap                      {central - federated.mean():+.4f}")
    print(f"federated spread         {federated.max() - federated.min():.4f}")
    inside = int((abs(federated - central) <= NOISE_FLOOR).sum())
    print(f"\n{inside} of {len(federated)} federated runs sit inside the "
          f"{NOISE_FLOOR} noise floor of the centralised baseline.")
    above = int((federated > central).sum())
    print(f"{above} of {len(federated)} are above it outright.")

## FedAvg against FedProx, by partition

The comparison RQ3 asks for, and the one place in the campaign where the partition
changes the answer.

The proximal term exists to stop a site drifting from the model it was given. Under
stratified partitions there is very little to drift from, because every hospital
holds the same class mix and only differs in size, so FedProx has nothing to
correct. Under the cohort partition it does.

In [ ]:
if SUMMARY_CSV.is_file():
    fed = pd.read_csv(SUMMARY_CSV).set_index("experiment")
    pairs = [
        ("2 hospitals, balanced", "test02", "test03"),
        ("3 hospitals, balanced", "test04", "test05"),
        ("4 hospitals, balanced", "test06", "test07"),
        ("4 hospitals, skewed 5:2:1:1", "test08", "test09"),
        ("3 hospitals, one cohort each", "test10", "test11"),
        ("3 hospitals, size-matched", "test12", "test13"),
    ]
    rows = []
    for label, avg, prox in pairs:
        if avg not in fed.index or prox not in fed.index:
            continue
        rows.append({"partition": label,
                     "fedavg": round(fed.loc[avg, "test_auc"], 4),
                     "fedprox": round(fed.loc[prox, "test_auc"], 4),
                     "prox_minus_avg": round(fed.loc[prox, "test_auc"]
                                             - fed.loc[avg, "test_auc"], 4)})
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
    print()
    print("Every one of these differences is inside the 0.067 noise floor, so none of")
    print("them is a result on its own. What is worth reporting is the DIRECTION and")
    print("where it is largest: FedProx gains most exactly where the heterogeneity is")
    print("real, and gains nothing where the sites are stratified copies of each other.")
else:
    print("no summary table yet")

## What this notebook produced

| path | what it is |
|---|---|
| `results/federated/all_experiments.csv` | the summary table, one row per experiment, scored on the same 268 patients |
| `results/federated/testNN_*/` | one folder per experiment: the aggregated global model, `test_metrics.json`, `predictions_test.csv`, and per-site `rounds.csv` |
| `deployment/logs/testNN/` | server, hospital and admin logs for that run, timestamped |

**What is still driven from `src/` and why**

`federation/client.py` and `federation/recipes.py`, because the hospitals are real
separate processes and the FLARE runtime imports the client script inside each of
them. A notebook cell cannot be imported by another process. The shell scripts that
start and stop the federation, because those processes have to outlive this kernel.

Everything else, the model, the data loading, the training loop and the metrics, is
the same code notebook 03 inlines, imported by both arms from `src/federated/common/`
so that a hyperparameter cannot differ between them.

**Reading the result honestly**

The federated runs land in the same band as the centralised baseline, and the gap is
smaller than the spread between two runs of the same configuration. That is the
answer to RQ1 and it is a positive one: the system works, and moving the training to
where the data is did not cost anything measurable on this task.

Where the partitions do bite is the cohort split, and that is the answer to RQ2. It
is also the only place where the FedAvg and FedProx difference points anywhere at
all, which is the answer to RQ3.